# 06 - Online recalibration

A plant does not stand still: substrate quality changes with the harvest, the
biology adapts, a digester is emptied. A model calibrated in spring drifts away
from the plant by autumn.

`OnlineCalibrator` is built for that case. It differs from the initial
calibration in two ways:

- it starts from the **currently used** parameters instead of a default,
- it **refuses** to move them far, or at all, when the evidence is thin.

Those guardrails are the point: an automatic recalibration that follows every
sensor glitch is worse than none.

This notebook runs the calibrator the way it would run next to a real plant: the
plant never restarts, and every recalibration continues from where the model stood.

## A plant that keeps running

Two copies of the demo plant run side by side. `plant` stands in for the real one
and produces the measurements, `model` is the digital twin we recalibrate. Both
first run 30 days on the standard feed, so they start from a settled digester and
not from the model's default start state. `advance()` keeps the state a run ends
in, so the next run continues from there.

Over three periods of five days the plant's hydrolysis gets slower: `k_hyd_ch`
drops from 2.0 to 1.5 to 1.0 (a smaller rate constant is a slower hydrolysis). The
value changes at the start of a period.

Every period also contains feed changes: a maize pulse, a day with grass silage. A
plant that has been fed the same mix for weeks shows almost nothing of its
hydrolysis rate, because the gas follows the feed whatever `k_hyd_ch` is. Only after
a change in the feed does the rate show, in how fast the gas follows.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from demo_plant import FEED_COLUMNS, add_measurement_noise, advance, build_demo_plant, feed_frame, simulate
from matplotlib.ticker import MaxNLocator

from pyadm1ode_calibration import OnlineCalibrator

WARMUP_DAYS = 30
PERIOD_DAYS = 5
DRIFT = [2.0, 1.5, 1.0]  # true k_hyd_ch per period: the hydrolysis gets slower
# Feed changes per period as (day, [maize, manure, grass]) in m3/d. Every period
# ends on the standard mix of 15 / 10 / 0.
FEED_CHANGES = [
    [(1, [30, 10, 0]), (2, [15, 10, 0]), (3, [15, 10, 15]), (4, [15, 10, 0])],
    [(1, [15, 10, 15]), (2, [15, 10, 0]), (3, [30, 10, 0]), (4, [15, 10, 0])],
    [(1, [30, 5, 0]), (2, [15, 10, 0]), (3, [22, 10, 8]), (4, [15, 10, 0])],
]

total_days = WARMUP_DAYS + PERIOD_DAYS * len(DRIFT)
plant = build_demo_plant(days=total_days)  # stands in for the real plant
model = build_demo_plant(days=total_days)  # the digital twin we recalibrate

# A common history: both settle on the standard feed with the value the model was
# last calibrated to.
start = pd.Timestamp("2024-01-01")
warmup = feed_frame(WARMUP_DAYS, start)
advance(plant, warmup, {"k_hyd_ch": DRIFT[0]})
advance(model, warmup, {"k_hyd_ch": DRIFT[0]})
start += pd.Timedelta(days=WARMUP_DAYS)

# The plant runs on through the three periods. A period's noisy outputs are what the
# operator has on the day of the recalibration.
rng = np.random.default_rng(10)
periods = []
for true_k, changes in zip(DRIFT, FEED_CHANGES):
    feed = feed_frame(PERIOD_DAYS, start, changes)
    outputs = advance(plant, feed, {"k_hyd_ch": true_k})
    periods.append(add_measurement_noise(feed, outputs, noise=0.02, rng=rng))
    start += pd.Timedelta(days=PERIOD_DAYS)
print("true k_hyd_ch per period:", DRIFT)

## Following the drift

Once per period the calibrator gets that period's measurements, the way it would
run in operation. Two things carry over from one round to the next:

- **the parameter**: a round starts from the value the previous one returned;
- **the state**: the calibrator simulates the period from the state `model` is in
  at its start. After the round, `model` runs through the period with the new
  value, so the next round starts where the twin actually stands.

`time_varying_feed=True` makes every simulation follow the feed hour by hour. By
default the calibrator applies the mean feed of the period, and the feed changes
would be lost.

In [ ]:
CHANNEL = "Q_gas"
current = {"k_hyd_ch": DRIFT[0]}
trajectory = [current["k_hyd_ch"]]
before_after = []  # simulation with the old and with the new value, per period

for i, period in enumerate(periods, start=1):
    result = OnlineCalibrator(model, verbose=False, time_varying_feed=True).calibrate(
        period,
        parameters=["k_hyd_ch"],
        current_parameters=current,
        objectives=[CHANNEL],
        max_parameter_change=0.20,
        variance_threshold=0.15,
        max_iterations=15,
        use_constraints=False,
    )
    # Both simulations start from the state the twin is in at the start of the period.
    before_after.append(
        tuple(
            np.asarray(simulate(model, period, values, time_varying_feed=True)[CHANNEL], dtype=float)
            for values in (current, result.parameters)
        )
    )
    current = dict(result.parameters)
    trajectory.append(current["k_hyd_ch"])
    # Hand the state over: the twin runs through the period with the new value.
    advance(model, period, current)
    print(f"period {i}: true {DRIFT[i - 1]:.2f} -> tracked {current['k_hyd_ch']:.3f}   ({result.message})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
# Round i returns trajectory[i] for the period in which the plant had DRIFT[i - 1].
ax.plot(range(len(trajectory)), trajectory, marker="o", label="tracked (after the round)")
ax.plot(range(1, len(DRIFT) + 1), DRIFT, marker="s", ls="--", color="tab:green", label="true (during the period)")
ax.set_xlabel("recalibration round")
# Rounds are counts, so half-integer ticks would be meaningless.
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_ylabel("k_hyd_ch")
ax.set_title("Parameter tracking a drifting plant")
ax.legend()
fig.tight_layout()

The tracked value lags behind: from 2.0 the true 1.5 is a change of 25 %, from the
tracked value the next 1.0 more than a third, and `max_parameter_change=0.20` lets a
round move by at most 20 %. That is the guardrail at work: if the plant stays at its
new value, the following rounds close the gap step by step, and a single noisy
period cannot pull the model far. Rounds 2 and 3 end exactly on that limit, which
is why Nelder-Mead reports the iteration limit there: it keeps pressing against the
bound.

## What the tracking does to the model output

The parameter plot shows the calibrator following the drift, but not whether the
*model* got better. The plot below puts the three periods on one time axis: the
measured `Q_gas`, and the simulation with the value a round **started from** and with
the one it **returned**, both from the twin's state at the start of the period. The
lower panel shows the feed.

In [ ]:
FEED_STYLE = {"Q_sub_maize": ("maize", "tab:orange"), "Q_sub_manure": ("manure", "tab:brown"),
              "Q_sub_grass": ("grass", "tab:green")}


def rmse(y, reference):
    return float(np.sqrt(np.mean((y - reference) ** 2)))


fig, (ax, ax_feed) = plt.subplots(
    2, 1, figsize=(11, 5.5), sharex=True, gridspec_kw={"height_ratios": [3, 1.2]}
)
for i, (period, (sim_before, sim_after)) in enumerate(zip(periods, before_after), start=1):
    t = period.data.index
    measured = period.data[CHANNEL].to_numpy()
    first = i == 1  # one legend entry per line
    ax.plot(t, measured, color="0.6", lw=0.8, label="measured" if first else None)
    ax.plot(t, sim_before, ls="--", color="tab:red", label="before recalibration" if first else None)
    ax.plot(t, sim_after, color="tab:blue", label="after recalibration" if first else None)
    for axis in (ax, ax_feed):
        axis.axvline(t[0], color="0.3", lw=0.6)
    ax.text(
        t[len(t) // 2], 0.97,
        f"period {i}: true {DRIFT[i - 1]:.2f}\nk_hyd_ch {trajectory[i - 1]:.2f} -> {trajectory[i]:.2f}",
        transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=8,
    )
    for column in FEED_COLUMNS:
        name, color = FEED_STYLE[column]
        ax_feed.step(t, period.data[column], where="post", color=color, label=name if first else None)

    print(
        f"period {i}: RMSE(Q_gas) {rmse(sim_before, measured):6.1f}"
        f" -> {rmse(sim_after, measured):6.1f} m3/d"
    )

ax.set_ylabel(f"{CHANNEL} [m3/d]")
ax.legend(fontsize=8, loc="lower right")
ax_feed.set_ylabel("feed [m3/d]")
ax_feed.legend(fontsize=8, loc="upper left", ncol=3)
ax_feed.tick_params(axis="x", rotation=30, labelsize=8)
fig.suptitle("Measurement against simulation, before and after each recalibration")
fig.tight_layout()

## Where this stops working

The tracking works here because the twin enters every period with a state that
matches the plant: the plant's kinetics change at the start of a period, right
where the last recalibration ended. Two situations break that, and the calibrator
alone cannot repair them:

- **A drift within the period.** One value per round has to explain a moving target
  and falls further behind.
- **A twin whose state has run away from the plant.** A model that ran for weeks
  with a wrong value carries a wrong state, for instance a wrong stock of
  undigested carbohydrates. The calibrator then prefers the value that fits that
  state, typically the old one.

Both call for estimating the state along with the parameters, for instance with a
Kalman filter as in
[PyADM1ODE_estimate](https://github.com/dgaida/PyADM1ODE_estimate).